# 7. The command line & configuration

Everything you can do in Python has a CLI equivalent, and every behaviour is configurable via
`ef.settings` (or an `EUROFLOOD_*` environment variable). This is a reference for both.

## The `euroflood` CLI

```bash
euroflood floods "Zutphen, Netherlands"                        # print a catalogue
euroflood floods "Zutphen, Netherlands" --start 2024 -o zutphen.geojson  # filter + write (CSV/GeoJSON/Parquet)
euroflood floods "Zutphen, Netherlands" --query "area_km2 > 1"  # a pandas .query() expression
euroflood floods "Zutphen, Netherlands" --shape bbox           # a clean bounding-box ROI
euroflood download "Zutphen, Netherlands" --start 2024 --out out/  # search + download + crop
euroflood hazard "Zutphen, Netherlands" -r 100 -r 500 --download --out hazard/
euroflood --json floods "Zutphen, Netherlands" | jq '.[0]'     # machine-readable JSON
euroflood floods "Zutphen, Netherlands" --dry-run              # show the plan, change nothing
```

Global flags: `-v/--verbose`, `-q/--quiet`, `--json`, `--no-color`, `--version`.

## Configuration in Python

`ef.settings` is a Pydantic model: inspect it, or read/write individual fields. Every field
also has an `EUROFLOOD_<NAME>` environment variable.

In [1]:
import euroflood as ef

print("index_mode      :", ef.settings.index_mode)
print("geocoder_backend:", ef.settings.geocoder_backend)
print("show_progress   :", ef.settings.show_progress)
print("geocode_cache   :", ef.settings.geocode_cache)
print("downloads go to :", ef.settings.output_dir.name + "/  (settings.output_dir)")

index_mode      : auto
geocoder_backend: online_first
show_progress   : True
geocode_cache   : True
downloads go to : outputs/  (settings.output_dir)


Override at runtime (equivalent to setting the env var before launch):

In [2]:
from pathlib import Path

ef.settings.output_dir = Path("results")  # or: export EUROFLOOD_OUTPUT_DIR=results
ef.settings.show_progress = True  # download/mirror bars when interactive
ef.settings.output_dir

PosixPath('results')

## Fast & offline

By default EuroFlood streams the published index over the network and geocodes place names
online (both cached after first use). For fully offline / HPC / reproducible runs:

```bash
export EUROFLOOD_GEOCODER_BACKEND=local   # resolve names from the offline NUTS dataset
export EUROFLOOD_OFFLINE=1                 # both collections cache-only, geocoder offline
euroflood mirror all --bbox 6.1 52 6.3 52.2 -r 100   # stage a region for offline use
```

…or pass a `bbox=` / `point=` to skip geocoding, and `EUROFLOOD_SHOW_PROGRESS=0` to silence
progress bars (e.g. in CI).

All of this is just configuration around the same query you already know:

In [3]:
ef.floods("Zutphen, Netherlands").head()

,collection,event_id,date,year,end_date,cluster_id,filename,download_url,area_km2,geometry
0,historic,4049036842,2019-03-04,2019,2019-04-08,333,WD_MERGE_2019-03-04---2019-04-08_duration_28_d...,https://jeodpp.jrc.ec.europa.eu/ftp/jrc-openda...,2.789,"POLYGON ((6.14039 52.1484, 6.14039 52.14835, 6..."
1,historic,1407437849,2024-02-05,2024,2024-03-11,345,WD_MERGE_2024-02-05---2024-03-11_duration_28_d...,https://jeodpp.jrc.ec.europa.eu/ftp/jrc-openda...,0.206,"POLYGON ((6.14039 52.1484, 6.14039 52.14835, 6..."
2,historic,1811386473,2018-01-22,2018,2018-03-05,358,WD_MERGE_2018-01-22---2018-03-05_duration_35_d...,https://jeodpp.jrc.ec.europa.eu/ftp/jrc-openda...,7.273,"POLYGON ((6.14039 52.1484, 6.14039 52.14835, 6..."
3,historic,2217388040,2024-01-01,2024,2024-02-19,334,WD_MERGE_2024-01-01---2024-02-19_duration_35_d...,https://jeodpp.jrc.ec.europa.eu/ftp/jrc-openda...,1.954,"POLYGON ((6.14039 52.1484, 6.14039 52.14835, 6..."
4,historic,3168514366,2019-12-30,2020,2020-02-10,371,WD_MERGE_2019-12-30---2020-02-10_duration_42_d...,https://jeodpp.jrc.ec.europa.eu/ftp/jrc-openda...,2.937,"POLYGON ((6.14039 52.1484, 6.14039 52.14835, 6..."
